This notebook formats GTEx data for use in this project, note that the sample data was stored in another repository (and prepared using another notebook: "") and the rpm counts are stored in google drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
!git clone https://github.com/Ignas12345/masters_project_helper_functions.git
sys.path.append('/content/masters_project_helper_functions')

Cloning into 'masters_project_helper_functions'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 46 (delta 19), reused 32 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 18.37 KiB | 9.18 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [3]:
import pandas as pd
import masters_project_helper_functions.utils as utils

In [4]:
def sum_by_index(df):
    df = df.copy()
    # Separate numerical and non-numerical columns
    num_cols = df.select_dtypes(include='number').columns
    non_num_cols = df.select_dtypes(exclude='number').columns

    # Sum numerical columns by index
    summed = df[num_cols].groupby(df.index).sum()

    # Take the first non-numerical row by index
    firsts = df[non_num_cols].groupby(df.index).first()

    # Combine both
    combined = pd.concat([summed, firsts], axis=1)

    return combined

In [5]:
gtex_data_path = '/content/drive/MyDrive/Magistro_projektas/Duomenys/GTEx/miRNA_TPM_matrix_PORTAL_2025_03_17.txt'
gtex_data_full = pd.read_csv(gtex_data_path, sep='\t', index_col=0)
gtex_data_full = gtex_data_full.T

In [6]:
gtex_samples_to_use_1  = pd.read_csv('https://raw.githubusercontent.com/Ignas12345/Magistras_knygutes_ir_duomenys/refs/heads/main/duomenys/TCGA_TGCT_ir_GTEx/training_samples_shuffled.csv', sep=',', index_col=0)
gtex_samples_to_use_2 = pd.read_csv('https://raw.githubusercontent.com/Ignas12345/Magistras_knygutes_ir_duomenys/refs/heads/main/duomenys/TCGA_TGCT_ir_GTEx/test_samples.csv', sep=',', index_col=0)
gtex_samples_to_use = pd.concat([gtex_samples_to_use_1, gtex_samples_to_use_2]).index

gtex_samples_to_use = gtex_samples_to_use[gtex_samples_to_use.str.startswith('GTEX')]
gtex_samples_to_use_truncated = gtex_samples_to_use.str[:18]
final_list_of_samples = [sample for sample in gtex_data_full.index if sample[:18] in gtex_samples_to_use_truncated]

gtex_data_to_use = gtex_data_full.loc[final_list_of_samples]
print(gtex_data_to_use.shape)

(111, 893)


In [7]:
small_rna_annotations_gtex_url = "https://storage.googleapis.com/adult-gtex/annotations/v10/small-RNA/smallRNA.filtered_annotated_031725.txt"
small_rna_annotations_gtex_df = pd.read_csv(small_rna_annotations_gtex_url, sep="\t")
small_rna_annotations_gtex_df.set_index("id", inplace=True)

rnas_to_drop = [row for row in small_rna_annotations_gtex_df.index if row not in gtex_data_to_use.columns]
# Drop the rows after identifying all rnas to drop
small_rna_annotations_trimmed = small_rna_annotations_gtex_df.drop(rnas_to_drop, inplace=False)
#drop duplicates of small_rna_annotations_trimmed:
small_rna_annotations_trimmed = small_rna_annotations_trimmed[~small_rna_annotations_trimmed.index.duplicated(keep='first')]

In a previous notebook, for TCGA-TGTC data, we use mir-16-5p as for endogenous normalization, however, for GTEx data this gene is missing (although other genes, close to this one in the list of genes are present!). So we test other possible alternatives for normalization genes, focusing on mirnas that were tested for stability in plasma in the literature: (https://www.sciencedirect.com/science/article/pii/S0039606015003499), such as RNU6 (turns out to be absent in the GTEx library also), mir-520d (not stable in tissue according to TCGA-TGCT data: https://github.com/Ignas12345/masters_project_data_and_notebooks/blob/main/Notebooks/pre_processing_methods/TCGA_TGCT_different_normalizations_and_filterings.ipynb). So other options could be either mir-29a or mir-191-5p. mir-191-5p was chosen because it is more stable in the TCGA-TGCT samples and was also tested in literature.

In [8]:
rnu6_indices = small_rna_annotations_gtex_df.loc['URS00001EC8D7'].index
#gtex_data_full['URS00001EC8D7'] <- gives an error

In [9]:
#for each row in data, the name of the row corresponds to an 'id' in mdata_df. so for each row, add a column 'mirbase_name' and 'MIMAT' by acessing those columns using the rowname of data and 'id' column of mdata_df:
gtex_data_merged = small_rna_annotations_trimmed[['MIMAT']].merge(gtex_data_to_use.T, left_index=True, right_index=True, how='left')
gtex_data_merged = gtex_data_merged[gtex_data_merged['MIMAT'].notna()]
gtex_data_merged.set_index('MIMAT', inplace=True)

gtex_data_final = sum_by_index(gtex_data_merged)

In [10]:
#check if the index of gtex data is unique:
gtex_data_final.index.is_unique

True

In [11]:
#get features from our tcga data
url_TCGA_mirna_rpm_counts = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_rpm.csv"
url_TCGA_TGCT_divisions_by_experiment = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/sample_annotations/TCGA_TGCT_divisions_by_experiment.csv"

TCGA_TGCT_divisions_by_experiment = pd.read_csv(url_TCGA_TGCT_divisions_by_experiment, index_col=0)
TCGA_mirna_rpm_counts = utils.format_data_frame(url_TCGA_mirna_rpm_counts, sep = ';', decimal = ',', transpose = True).loc[TCGA_TGCT_divisions_by_experiment.index]
TCGA_mirna_rpm_counts = utils.initial_pre_processing_pipeline(TCGA_mirna_rpm_counts)

tcga_features = TCGA_mirna_rpm_counts.columns
tcga_features_truncated = tcga_features.str[-12:]

reading df from url: https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_rpm.csv
nan values filled with 0
column names truncated using: slice(None, 15, None)
df transposed
final shape of df: (156, 3689)

initial shape: (137, 3689)
shape after collapsing: (137, 2212)
initial shape: (137, 2212)
shape after filtering: (137, 2191)


In [12]:
# Create a dictionary mapping truncated TCGA feature names to full names
tcga_feature_map = dict(zip(tcga_features_truncated, tcga_features))

# Create a new column 'feature_name' in gtex_data_final
# Map the current index (MIMAT) to the full TCGA feature name using the dictionary
gtex_data_final['feature_name'] = gtex_data_final.index.map(tcga_feature_map)

In [13]:
gtex_data_reformatted = gtex_data_final.dropna(inplace=False).set_index('feature_name', inplace=False).T
print(gtex_data_reformatted.shape)

(111, 788)


In [14]:
#filter columns that are empty:
gtex_data_reformatted = gtex_data_reformatted.loc[:, gtex_data_reformatted.sum(axis=0) > 0]
print(gtex_data_reformatted.shape)

(111, 741)


In [15]:
#check if 371a-3p is present
gtex_data_reformatted['hsa-mir-371a, mature,MIMAT0000723']

,"hsa-mir-371a, mature,MIMAT0000723"
GTEX-11EQ9-1926-SM-EVLQ3,88.384624
GTEX-11LCK-2326-SM-DEUQS,111.864661
GTEX-11NSD-1026-SM-EBBEU,91.842839
GTEX-11O72-0726-SM-EBB7R,43.042538
GTEX-11P7K-1026-SM-EVLQP,28.225497
...,...
GTEX-ZU9S-2426-SM-GMKHW,206.613278
GTEX-ZVP2-1026-SM-GH6FX,99.858134
GTEX-ZVTK-0126-SM-EV6TS,116.546893
GTEX-ZYT6-2726-SM-EVLK1,101.641816


In [16]:
#save to csv:
gtex_data_reformatted.to_csv('gtex_data_reformatted.csv', sep = ';')

In [17]:
#perform stability analysis for houskeeping mirnas:

df_to_use = gtex_data_reformatted.copy()
#This is to calculate the relative variance of each feature (ChatGPT's function)
df_for_stability_analysis = df_to_use.copy()
#df_for_stability_analysis = np.log2(df_for_stability_analysis + 1)  # Log-transform if needed
#flter features with stds. that are smaller than 1:
#df_for_stability_analysis = df_for_stability_analysis.loc[:, df_for_stability_analysis.std() > 5]
# Calculate coefficient of variation per miRNA
cv = df_for_stability_analysis.std(axis=0) / df_for_stability_analysis.mean(axis=0)

# Rank by stability (lower CV = more stable)
stable_mirnas = cv.sort_values()
stable_mirnas.head(21)

,0
feature_name,
"hsa-mir-26a-1, mature,MIMAT0000082",0.225808
"hsa-mir-152, mature,MIMAT0000438",0.229723
"hsa-mir-339, mature,MIMAT0004702",0.238133
"hsa-mir-24-2, mature,MIMAT0004497",0.249280
"hsa-mir-125b-2, mature,MIMAT0004603",0.249906
"hsa-mir-26b, mature,MIMAT0000083",0.252654
"hsa-mir-140, mature,MIMAT0004597",0.253215
"hsa-mir-361, mature,MIMAT0000703",0.255924
"hsa-mir-345, mature,MIMAT0000772",0.258546
